# Course 6 Lab — Model Gateways and Inference Economics

**Thesis:** model access is a policy and economics boundary balancing task quality,
latency, availability, privacy, safety, and cost—not one provider SDK call.

This credential-free lab compares direct, cascade, fallback, and delayed-hedge policies
for Northstar's underwriting summaries. All providers are deterministic fictional
simulators, so the notebook proves gateway behavior and metric arithmetic rather than live
model quality or vendor pricing.

## 1. Scenario and success contract

Northstar has an inexpensive fast model and a stronger slower model. Restricted data must
stay in Canada. A successful task returns the correct labelled decision through an eligible
model, passes schema and case binding, meets the deadline, and remains inside hard token and
cost budgets.

We will select the least expensive policy that satisfies a compliant-success floor and
p95-latency ceiling. HTTP success and fluent prose do not define success.

## 2. Safety and reproducibility boundaries

- No provider account, API key, cloud resource, network call, or paid inference is used.
- Cost units and model behavior are fictional and deterministic.
- Authenticated tenant and region come from application context, never the prompt.
- Every provider result crosses the same schema and resource-binding validator.
- Failed and losing calls remain in cost and attempt accounting.
- Observable route decisions are shown; private chain-of-thought is not requested.

## 3. Learning objectives

By the end you will be able to normalize provider contracts, enforce model eligibility,
classify fallback-safe errors, reserve budgets and quotas, design safe cache keys, measure
cascade and hedge amplification, calculate cost per successful compliant task, and choose a
route from a quality-latency-cost frontier.

## 4. Architecture before implementation

![Governed model gateway decision path from eligibility and caching through routing, validation, fallback, settlement, and terminal results.](assets/gateway-decision-path.svg)

Provider selection happens only inside the eligible set. A common response shape does not
erase provider capability, retention, safety, streaming, or quality differences.

In [1]:
from dataclasses import replace

from lab import (
    DataClass,
    ErrorKind,
    FaultPlan,
    GatewayBoundaryError,
    GatewayStatus,
    ProviderBehavior,
    ProviderFault,
    QuotaLedger,
    SpendLedger,
    build_demo_environment,
    demo_request,
    deployment_decision_map,
    evaluate_policy,
    pareto_frontier,
    recommend_policy,
    report_rows,
    threshold_sweep,
)

print("Course 6 deterministic gateway ready")

Course 6 deterministic gateway ready


## 5. Inspect the normalized model catalogue

The catalogue contains stable application selection facts: versions, regions, data
classes, capabilities, context limits, relative price metadata, and retention mode. A real
platform would reconcile this registry with provider lifecycle and contract data.

In [2]:
env = build_demo_environment()
for profile in env.profiles.values():
    print(
        profile.model_id,
        "provider=", profile.provider,
        "regions=", sorted(profile.regions),
        "capabilities=", sorted(profile.capabilities),
        "retention=", profile.retention_mode,
    )

economy-ca provider= provider-a regions= ['ca-central-1'] capabilities= ['streaming', 'structured-output'] retention= regional-no-training
premium-ca provider= provider-b regions= ['ca-central-1'] capabilities= ['long-context', 'streaming', 'structured-output'] retention= regional-no-training
premium-us provider= provider-c regions= ['us-east-1'] capabilities= ['long-context', 'streaming', 'structured-output'] retention= us-no-training


The three deployments are deliberately not equivalent. `premium-us` has a useful latency
profile but is outside the Canadian processing boundary, so it is never a valid fallback
for the default authenticated context.

## 6. Baseline A — direct economy model

Always begin with a simple baseline. The economy route gives us minimum cost and latency,
but the labelled set—not the model name—determines whether quality is sufficient.

In [3]:
economy_env = build_demo_environment()
economy_report = evaluate_policy(
    economy_env.gateway,
    economy_env.context,
    economy_env.policies["economy-direct"],
    economy_env.cases,
)
report_rows([economy_report])

[{'policy': 'economy-direct',
  'success_rate': 0.5,
  'compliant_success_rate': 0.5,
  'p95_latency_ms': 95.0,
  'total_cost_units': 1.094,
  'cost_per_compliant_success': 0.547,
  'provider_calls': 4,
  'escalations': 0}]

All requests complete at the transport layer, but only half match the labelled decision.
This is why availability and HTTP status cannot stand in for task success.

## 7. Baseline B — direct premium model

The premium baseline is the quality reference for this tiny fixture. It is not assumed to
be universally correct; it simply happens to match all four deterministic labels.

In [4]:
premium_env = build_demo_environment()
premium_report = evaluate_policy(
    premium_env.gateway,
    premium_env.context,
    premium_env.policies["premium-direct"],
    premium_env.cases,
)
report_rows([premium_report])

[{'policy': 'premium-direct',
  'success_rate': 1.0,
  'compliant_success_rate': 1.0,
  'p95_latency_ms': 210.0,
  'total_cost_units': 11.84,
  'cost_per_compliant_success': 2.96,
  'provider_calls': 4,
  'escalations': 0}]

Premium direct reaches the fixture's quality floor at substantially higher cost and p95
latency. Routing is justified only if it finds a better operating point without hiding
difficult or restricted slices.

## 8. Trace a simple cascade acceptance

A cascade first calls the economy model, validates the output, then asks a separate
calibrated acceptance estimator whether to stop or escalate. Provider self-confidence is
carried in the response but is not the acceptance gate.

In [5]:
cascade_env = build_demo_environment()
simple = cascade_env.gateway.infer(
    cascade_env.context,
    demo_request(request_id="notebook-simple"),
    cascade_env.policies["quality-cascade"],
)
print(simple.terminal_reason, simple.selected_model, simple.total_cost_units)
print(simple.attempts)
assert simple.provider_calls == 1
assert simple.attempts[0].quality_score == 0.96

CASCADE_ACCEPTED_PRIMARY economy-ca 0.206
(AttemptRecord(attempt_id='notebook-simple:1:economy-ca', logical_request_id='notebook-simple', model_id='economy-ca', provider='provider-a', start_ms=0, latency_ms=75, cost_units=0.206, usage=NormalizedUsage(input_tokens=1000, cached_input_tokens=600, output_tokens=120), error_kind=None, provider_request_id='provider-a:notebook-simple:1', accepted=True, quality_score=0.96),)


The gate accepts the simple economy result, so the premium cost and serial latency are
avoided. The gate score is observable evidence, not a correctness certificate.

## 9. Trace a complex-case escalation

The economy output is schema-valid but receives a low acceptance score. The gateway then
reserves the premium call, invokes it, validates it independently, and reports both calls.

In [6]:
complex_result = cascade_env.gateway.infer(
    cascade_env.context,
    demo_request("case-complex", request_id="notebook-complex"),
    cascade_env.policies["quality-cascade"],
)
print(complex_result.terminal_reason, complex_result.selected_model)
for attempt in complex_result.attempts:
    print(attempt)
assert complex_result.provider_calls == 2
assert complex_result.output.decision == "refer"

CASCADE_ESCALATED_LOW_SCORE premium-ca
AttemptRecord(attempt_id='notebook-complex:1:economy-ca', logical_request_id='notebook-complex', model_id='economy-ca', provider='provider-a', start_ms=0, latency_ms=95, cost_units=0.296, usage=NormalizedUsage(input_tokens=1000, cached_input_tokens=0, output_tokens=120), error_kind=None, provider_request_id='provider-a:notebook-complex:1', accepted=False, quality_score=0.42)
AttemptRecord(attempt_id='notebook-complex:2:premium-ca', logical_request_id='notebook-complex', model_id='premium-ca', provider='provider-b', start_ms=95, latency_ms=210, cost_units=3.2, usage=NormalizedUsage(input_tokens=1000, cached_input_tokens=0, output_tokens=150), error_kind=None, provider_request_id='provider-b:notebook-complex:1', accepted=True, quality_score=None)


Escalation adds both the economy cost and serial latency. A cascade is not a free selector;
its economics depend on the escalation rate and gate cost.

## 10. Threshold sensitivity experiment

Sweep the acceptance threshold rather than choosing it by intuition. A low threshold accepts
more cheap output; a high threshold approaches premium quality while retaining the probe
cost and serial path.

In [7]:
sweep = threshold_sweep((0.40, 0.80, 0.95))
report_rows(sweep)

[{'policy': 'cascade-0.40',
  'success_rate': 0.5,
  'compliant_success_rate': 0.5,
  'p95_latency_ms': 95.0,
  'total_cost_units': 1.094,
  'cost_per_compliant_success': 0.547,
  'provider_calls': 4,
  'escalations': 0},
 {'policy': 'cascade-0.80',
  'success_rate': 0.75,
  'compliant_success_rate': 0.75,
  'p95_latency_ms': 305.0,
  'total_cost_units': 4.294,
  'cost_per_compliant_success': 1.431333,
  'provider_calls': 5,
  'escalations': 1},
 {'policy': 'cascade-0.95',
  'success_rate': 1.0,
  'compliant_success_rate': 1.0,
  'p95_latency_ms': 305.0,
  'total_cost_units': 10.694,
  'cost_per_compliant_success': 2.6735,
  'provider_calls': 7,
  'escalations': 3}]

On this fixture the stricter threshold improves compliant success and increases cost,
provider calls, and p95 latency. The right threshold is a release-policy choice tied to
representative held-out labels.

## 11. Failure injection — router drift

The drift case is intentionally dangerous: the gate assigns a high score to the wrong
economy decision. The gateway is functioning as configured, but the routing model is no
longer reliable on that slice.

In [8]:
drift_env = build_demo_environment()
drift = drift_env.gateway.infer(
    drift_env.context,
    demo_request("case-drift", request_id="notebook-drift"),
    drift_env.policies["quality-cascade"],
)
print(drift.output, drift.attempts[0].quality_score)
assert drift.output.decision == "approve"  # labelled truth is decline
assert drift.attempts[0].quality_score == 0.92

ModelOutput(case_id='case-drift', decision='approve', summary='Normalized recommendation for case-drift', confidence=0.88, schema_version='underwriting-summary/v1') 0.92


Mitigation is not a prompt saying “be accurate.” Maintain labels, calibration curves,
difficult slices, route/model version telemetry, drift thresholds, shadow evaluation, and a
rollback to a known route.

## 12. Reliability fallback after throttling

Fallback addresses classified availability failure, not quality. Here the economy provider
throttles, so the gateway uses the already-admitted Canadian premium model.

In [9]:
throttle_faults = FaultPlan()
throttle_faults.add(
    "economy-ca",
    "case-simple",
    ProviderFault(ErrorKind.THROTTLED, 35, "simulated 429"),
)
fallback_env = build_demo_environment(faults=throttle_faults)
fallback = fallback_env.gateway.infer(
    fallback_env.context,
    demo_request(request_id="notebook-throttle"),
    fallback_env.policies["reliability-fallback"],
)
print(fallback.terminal_reason, [a.model_id for a in fallback.attempts])
assert fallback.selected_model == "premium-ca"

FALLBACK_SUCCESS ['economy-ca', 'premium-ca']


Both attempts remain in the trace. In production, include any billed tokens from the failed
attempt and correlate the provider request ID with invoices and incident data.

## 13. Terminal authentication failure

A bad provider credential is a platform defect. Trying another provider can conceal broken
secret rotation or an attack, so authentication is not in the retryable error set.

In [10]:
auth_faults = FaultPlan()
auth_faults.add(
    "economy-ca",
    "case-simple",
    ProviderFault(ErrorKind.AUTHENTICATION, 10, "simulated bad credential"),
)
auth_env = build_demo_environment(faults=auth_faults)
auth_result = auth_env.gateway.infer(
    auth_env.context,
    demo_request(request_id="notebook-auth"),
    auth_env.policies["reliability-fallback"],
)
print(auth_result.status, auth_result.attempts)
assert auth_result.status is GatewayStatus.FAILED
assert auth_result.provider_calls == 1

failed (AttemptRecord(attempt_id='notebook-auth:1:economy-ca', logical_request_id='notebook-auth', model_id='economy-ca', provider='provider-a', start_ms=0, latency_ms=10, cost_units=0.0, usage=NormalizedUsage(input_tokens=0, cached_input_tokens=0, output_tokens=0), error_kind=<ErrorKind.AUTHENTICATION: 'authentication'>, provider_request_id='provider-a:notebook-auth:1', accepted=False, quality_score=None),)


## 14. Safety refusal is not provider shopping

A safety refusal has different semantics from throttling. The default route stops; a product
may define a reviewed human or policy path, but arbitrary fallback would be a safety bypass.

In [11]:
safety_faults = FaultPlan()
safety_faults.add(
    "economy-ca",
    "case-simple",
    ProviderFault(ErrorKind.SAFETY_REFUSAL, 20, "simulated refusal"),
)
safety_env = build_demo_environment(faults=safety_faults)
safety_result = safety_env.gateway.infer(
    safety_env.context,
    demo_request(request_id="notebook-safety"),
    safety_env.policies["reliability-fallback"],
)
print(safety_result.status, safety_result.provider_calls)
assert safety_result.provider_calls == 1

failed 1


## 15. Residency denial on a tempting fallback

Simulate a Canadian provider outage and configure the faster US premium model as fallback.
The gateway must reauthorize every candidate, not treat emergency routing as permission.

In [12]:
residency_faults = FaultPlan()
residency_faults.add(
    "economy-ca",
    "case-restricted",
    ProviderFault(ErrorKind.UNAVAILABLE, 30, "simulated outage"),
)
residency_env = build_demo_environment(faults=residency_faults)
unsafe_route = replace(
    residency_env.policies["reliability-fallback"],
    secondary_model="premium-us",
)
residency_result = residency_env.gateway.infer(
    residency_env.context,
    demo_request(
        "case-restricted",
        request_id="notebook-residency",
        data_classification=DataClass.RESTRICTED,
    ),
    unsafe_route,
)
print(residency_result.status, residency_result.terminal_reason)
assert residency_result.terminal_reason == ErrorKind.RESIDENCY_DENIED

blocked residency_denied


Availability is calculated over eligible providers, not every reachable endpoint. The
denied fallback is valid work blocked, not a forbidden outcome—the policy prevented the
exposure.

## 16. Output validation before fallback

Provider-neutral JSON can still be malformed or refer to the wrong case. Inject a missing
decision field and observe validation before the fallback is accepted.

In [13]:
output_env = build_demo_environment()
output_env.gateway.provider.behaviors[("economy-ca", "case-simple")] = ProviderBehavior(
    "approve", 70, 100, 0.9, malformed=True
)
output_result = output_env.gateway.infer(
    output_env.context,
    demo_request(request_id="notebook-output"),
    output_env.policies["reliability-fallback"],
)
print(output_result.attempts)
assert output_result.attempts[0].error_kind is ErrorKind.OUTPUT_INVALID
assert output_result.selected_model == "premium-ca"

(AttemptRecord(attempt_id='notebook-output:1:economy-ca', logical_request_id='notebook-output', model_id='economy-ca', provider='provider-a', start_ms=0, latency_ms=70, cost_units=0.28, usage=NormalizedUsage(input_tokens=1000, cached_input_tokens=0, output_tokens=100), error_kind=<ErrorKind.OUTPUT_INVALID: 'output_invalid'>, provider_request_id='provider-a:notebook-output:1', accepted=False, quality_score=None), AttemptRecord(attempt_id='notebook-output:2:premium-ca', logical_request_id='notebook-output', model_id='premium-ca', provider='provider-b', start_ms=70, latency_ms=210, cost_units=2.24, usage=NormalizedUsage(input_tokens=1000, cached_input_tokens=600, output_tokens=150), error_kind=None, provider_request_id='provider-b:notebook-output:1', accepted=True, quality_score=None))


The failed output still incurred cost. Structured-output features reduce parsing failure;
application code still verifies semantic bindings and quality.

## 17. Hard budget reservation

Configure a cap that can admit the economy attempt but not the worst-case premium fallback.
The second call is blocked before inference rather than discovered on the invoice.

In [14]:
budget_faults = FaultPlan()
budget_faults.add(
    "economy-ca",
    "case-simple",
    ProviderFault(ErrorKind.UNAVAILABLE, 25, "simulated outage"),
)
budget_env = build_demo_environment(faults=budget_faults)
tight_route = replace(
    budget_env.policies["reliability-fallback"],
    max_cost_units=3.0,
)
budget_result = budget_env.gateway.infer(
    budget_env.context,
    demo_request(request_id="notebook-budget"),
    tight_route,
)
print(budget_result.status, budget_result.terminal_reason)
assert budget_result.terminal_reason == ErrorKind.BUDGET_EXHAUSTED

blocked budget_exhausted


Reservation is essential under concurrency. A check against completed spend alone lets many
simultaneous requests pass before any of them reports actual usage.

In [15]:
spend = SpendLedger()
spend.reserve("logical-request", "attempt-1", 6.0, 10.0)
try:
    spend.reserve("logical-request", "attempt-2", 6.0, 10.0)
except GatewayBoundaryError as error:
    print("second reservation denied:", error.kind)
    assert error.kind is ErrorKind.BUDGET_EXHAUSTED

second reservation denied: budget_exhausted


## 18. Token quota reservation

Quotas and budgets are different controls. Reserve the maximum token use for admitted
in-flight requests, then settle actual usage.

In [16]:
quota = QuotaLedger({"northstar": 2_000})
quota.reserve("northstar", "minute-1", "attempt-1", 1_200)
try:
    quota.reserve("northstar", "minute-1", "attempt-2", 1_200)
except GatewayBoundaryError as error:
    print("second token reservation denied:", error.kind)
    assert error.kind is ErrorKind.QUOTA_EXHAUSTED

second token reservation denied: quota_exhausted


## 19. Validated application response cache

Enable response caching for an identical tenant-, prompt-, route-, policy-, and model-
version-scoped request. The second call returns the previously validated output.

In [17]:
cache_env = build_demo_environment()
cache_route = replace(cache_env.policies["economy-direct"], use_response_cache=True)
cache_request = demo_request(request_id="notebook-cache")
first = cache_env.gateway.infer(cache_env.context, cache_request, cache_route)
second = cache_env.gateway.infer(cache_env.context, cache_request, cache_route)
print("first", first.cache_hit, first.total_cost_units)
print("second", second.cache_hit, second.total_cost_units)
assert second.cache_hit and second.provider_calls == 0

first False 0.206
second True 0.0


This is not provider prompt caching. Prompt/KV caching reuses prefix computation but still
generates a new response. Response caching returns an earlier validated output and therefore
needs stronger authorization, freshness, versioning, and invalidation.

## 20. Cache isolation and version invalidation

A different tenant must miss even with identical content. A changed route generation must
also miss so a policy rollout cannot silently reuse an old decision.

In [18]:
other_tenant = cache_env.gateway.infer(
    cache_env.other_tenant_context,
    cache_request,
    cache_route,
)
changed_route = replace(cache_route, policy_version="route-v3")
changed = cache_env.gateway.infer(cache_env.context, cache_request, changed_route)
print("other tenant hit:", other_tenant.cache_hit)
print("changed route hit:", changed.cache_hit)
assert not other_tenant.cache_hit and not changed.cache_hit

other tenant hit: False
changed route hit: False


## 21. Delayed hedging: latency versus total work

The premium primary takes 210 ms. After 100 ms, the economy hedge starts and completes at
195 ms. The user sees 195 ms, but both calls are billed and the faster answer may be wrong.

In [19]:
hedge_env = build_demo_environment()
hedge = hedge_env.gateway.infer(
    hedge_env.context,
    demo_request("case-complex", request_id="notebook-hedge"),
    hedge_env.policies["latency-hedge"],
)
print(
    "winner=", hedge.selected_model,
    "wall_ms=", hedge.latency_ms,
    "calls=", hedge.provider_calls,
    "cost=", hedge.total_cost_units,
    "decision=", hedge.output.decision,
)
assert hedge.provider_calls == 2 and hedge.latency_ms == 195

winner= economy-ca wall_ms= 195 calls= 2 cost= 3.496 decision= approve


Hedging is a latency technique, not a quality strategy. It must be evaluated against a
product rule for which successful response can win. Counting only the winner would hide
capacity amplification and a potential quality regression.

## 22. Deadline failure still has economics

The application deadline is 50 ms while the economy call takes 75 ms. The output is not
returned as success, but the work was performed and remains in cost accounting.

In [20]:
deadline_env = build_demo_environment()
deadline = deadline_env.gateway.infer(
    deadline_env.context,
    demo_request(request_id="notebook-deadline", deadline_ms=50),
    deadline_env.policies["economy-direct"],
)
print(deadline.status, deadline.total_cost_units, deadline.attempts)
assert deadline.status is GatewayStatus.FAILED
assert deadline.total_cost_units > 0

failed 0.206 (AttemptRecord(attempt_id='notebook-deadline:1:economy-ca', logical_request_id='notebook-deadline', model_id='economy-ca', provider='provider-a', start_ms=0, latency_ms=75, cost_units=0.206, usage=NormalizedUsage(input_tokens=1000, cached_input_tokens=600, output_tokens=120), error_kind=<ErrorKind.DEADLINE_EXCEEDED: 'deadline_exceeded'>, provider_request_id='provider-a:notebook-deadline:1', accepted=False, quality_score=None),)


## 23. Evaluate complete policy alternatives

Use fresh environments so cache, faults, budgets, and quota state do not contaminate the
comparison. Each report uses the same four labelled cases and explicit denominators.

In [21]:
policy_names = (
    "economy-direct",
    "premium-direct",
    "quality-cascade",
    "latency-hedge",
)
reports = []
for name in policy_names:
    policy_env = build_demo_environment()
    reports.append(
        evaluate_policy(
            policy_env.gateway,
            policy_env.context,
            policy_env.policies[name],
            policy_env.cases,
        )
    )
report_rows(reports)

[{'policy': 'economy-direct',
  'success_rate': 0.5,
  'compliant_success_rate': 0.5,
  'p95_latency_ms': 95.0,
  'total_cost_units': 1.094,
  'cost_per_compliant_success': 0.547,
  'provider_calls': 4,
  'escalations': 0},
 {'policy': 'premium-direct',
  'success_rate': 1.0,
  'compliant_success_rate': 1.0,
  'p95_latency_ms': 210.0,
  'total_cost_units': 11.84,
  'cost_per_compliant_success': 2.96,
  'provider_calls': 4,
  'escalations': 0},
 {'policy': 'quality-cascade',
  'success_rate': 0.75,
  'compliant_success_rate': 0.75,
  'p95_latency_ms': 305.0,
  'total_cost_units': 4.294,
  'cost_per_compliant_success': 1.431333,
  'provider_calls': 5,
  'escalations': 1},
 {'policy': 'latency-hedge',
  'success_rate': 0.5,
  'compliant_success_rate': 0.5,
  'p95_latency_ms': 195.0,
  'total_cost_units': 12.934,
  'cost_per_compliant_success': 6.467,
  'provider_calls': 8,
  'escalations': 4}]

`completed` and `correct` are intentionally separate. The cost denominator is compliant
correct completion, so inexpensive wrong answers do not appear as economic wins.

## 24. Pareto frontier and release constraints

The Pareto frontier removes policies that are no better on success, p95 latency, and cost.
Release constraints then select among the remaining candidates.

In [22]:
print("frontier:", pareto_frontier(reports))
recommendation = recommend_policy(
    reports,
    minimum_compliant_success_rate=0.95,
    maximum_p95_latency_ms=250,
)
print(recommendation)
assert recommendation.selected_policy == "premium-direct"

frontier: ('economy-direct', 'premium-direct', 'quality-cascade')
Recommendation(selected_policy='premium-direct', reason='LOWEST_COST_PER_SUCCESSFUL_COMPLIANT_TASK_WITHIN_CONSTRAINTS', eligible_policies=('premium-direct',))


On this fixture, only premium direct meets both constraints. If the latency ceiling were
tighter, no policy would qualify; the honest decision would be to redesign or renegotiate
the objective, not weaken the metric silently.

## 25. Current tooling decision map

A product feature list is not an architecture decision. Compare who owns identity, policy,
budget state, retry loops, output validation, evaluation, operation, and provider exit.

In [23]:
deployment_decision_map()

({'option': 'direct provider SDK',
  'strong_fit': 'one bounded workload and provider',
  'still_own': 'policy, budgets, retries, evaluation, provider migration'},
 {'option': 'Amazon Bedrock APIs / prompt routing',
  'strong_fit': 'AWS-governed multi-model access and supported managed routing',
  'still_own': 'application eligibility, workload evaluation, cost attribution'},
 {'option': 'LiteLLM gateway',
  'strong_fit': 'OpenAI-shaped multi-provider proxy, routing, limits, spend controls',
  'still_own': 'deployment, state dependencies, policy correctness, upgrade testing'},
 {'option': 'cloud API management / managed AI gateway',
  'strong_fit': 'central platform identity, quotas, network policy, governance',
  'still_own': 'model semantics, routing evidence, output validation'},
 {'option': 'Kubernetes or Envoy AI gateway',
  'strong_fit': 'self-hosted inference pools and platform-native traffic policy',
  'still_own': 'model-server signals, capacity, control-plane and data-plane o

Direct SDKs minimize moving parts. Bedrock adds managed multi-model APIs and routing within
documented constraints. LiteLLM supplies a multi-provider proxy and router. Cloud API
management centralizes platform controls. Kubernetes/Envoy inference gateways serve teams
operating model data planes. Every option still needs application workload evidence.

## 26. Production upgrade plan

| Lab simplification | Production requirement |
|---|---|
| in-memory policy | signed/versioned configuration with staged rollout and rollback |
| fictional profiles | reconciled provider catalogue, contract and lifecycle data |
| local reservations | atomic distributed budget, token, and concurrency state |
| deterministic adapters | provider contract tests, streaming, cancellation, typed errors |
| fixed cases | representative labelled replay, slices, human review, drift monitoring |
| local cache | encrypted scoped store, TTL, invalidation, deletion, stampede control |
| simulated latency | end-to-end p50/p95/p99 plus provider and gateway spans |
| one process | multi-zone capacity, bulkheads, load tests, game days, incident ownership |

Preserve the trusted boundary when adding live adapters: providers propose output; the
application owns authorization, eligibility, budgets, validation, and release evidence.

## 27. Portfolio deliverables

1. Routing policy covering eligibility, error classes, retry, fallback, hedge, cache, and
   terminal behavior.
2. Gateway ADR comparing direct, managed, open-source, and custom options.
3. Quality-latency-cost frontier with correct metric populations and slices.
4. Sensitivity report for thresholds, prices, token mix, drift, outages, and quotas.

## 28. Exercises

**Implementation**

1. Add a vision capability and prove that a text-only fallback is ineligible.
2. Add a fixed gateway fee and cache-write pricing to the economics.
3. Normalize a mid-stream error and cancel losing provider work.

**Diagnosis**

4. Add client and SDK retries, then expose worst-case amplification.
5. Simulate a correlated identity outage affecting every provider.
6. Shift the case mix and show threshold regret.

**Architecture judgment**

7. Compare Bedrock prompt routing, Foundry model router, LiteLLM, and a direct adapter.
8. Decide whether the latency objective justifies hedging under the token quota.
9. Define the Course 7 SLO and telemetry contract from these attempt records.

## 29. Summary

A production model gateway is a governed inference boundary. It admits only eligible model
paths, classifies failures, bounds total work, validates every result, attributes every
attempt, and chooses a routing policy from representative evidence. Normalization reduces
integration duplication; it does not make models, providers, regions, or safety behavior
equivalent.

The next course turns these request, attempt, route, cost, latency, cache, and error records
into correlated telemetry, SLOs, alerts, failure game days, and incident response.